# Notebook compare the SSLimPy response functions to Sims

## Initial Setup

In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline
from matplotlib import cm, colors
from matplotlib.ticker import LogLocator, FuncFormatter

In [ ]:
# seaborn.set_theme(rc={'axes.edgecolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black',})
import niceplots.utils as nicepl
nicepl.initPlot()

In [ ]:
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
import SSLimPy.LIMsurvey.power_spectrum as spobs

## Choose model parameters and save them in dictionaries

In [ ]:
covfile = np.loadtxt("/home/sefa/Desktop/LIM-Code/clusterdata/342461_jackknife.cov")
kc_BB = pkfile[1:,0]
cov_BB = covfile[1:, 1:]
corr = cov_BB / np.sqrt(np.outer(
    np.diag(cov_BB), np.diag(cov_BB)
))

In [ ]:
k = pkfile[1:,0]   # 1D vector, already log-spaced

# create edges for pcolormesh
logk = np.log10(k)
d = np.diff(logk)/2
k_edges = 10**np.concatenate(([logk[0]-d[0]], logk[:-1]+d, [logk[-1]+d[-1]]))

fig, ax = plt.subplots()

# pcolormesh takes edges × edges
mesh = ax.pcolormesh(k_edges, k_edges, corr, shading='auto')

ax.set_xscale('log')
ax.set_yscale('log')

plt.xlabel(r"$k_1\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$k_2\,[h\,\mathrm{Mpc}^{-1}]$")
plt.title("Big Box Correlation Matrix")

plt.colorbar(mesh, label="Correlation")
plt.show()

In [ ]:
sqfile = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_346009_subdivided_JK.npz")

s1 = sqfile["Cov"][0, :, :]
s1 = s1[~np.isnan(s1)]
s1 = s1.reshape((17,17))
ds1 = np.diag(s1)
corr1 = s1 / np.sqrt(np.outer(ds1, ds1))
plt.imshow(corr1)

In [ ]:
Covs = sqfile["Cov"]
dTb = sqfile["deltab"]
k = np.sqrt(sqfile["kedges"][1:] * sqfile["kedges"][:-1])

plt.scatter(k, Covs[15, :, 5])
plt.loglog()
plt.xlabel("$k\,[h\,\mathrm{Mpc}^{-1}]$")

In [ ]:
dCov = np.array([np.diag(Covs[i, :, :]) for i in range(64)])

# Set up colormap normalization
norm = colors.Normalize(vmin=np.min(dTb), vmax=np.max(dTb))
cmap = cm.cividis
sm = cm.ScalarMappable(norm=norm, cmap=cmap)

# Plot
fig, ax = plt.subplots()
for i in range(dCov.shape[0]):
    color = cmap(norm(dTb[i]))
    ax.loglog(k[:], dCov[i, :], color=color)

plt.loglog(kc_BB, np.diag(cov_BB) * 4**3, "k--")

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")
plt.xlabel("$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel("$Cov_{ii}\,[\mu\mathrm{K}^4\,\mathrm{Mpc}^{6}]$")

In [ ]:
# second = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_189352_CII_A_Response_NSUB8.npz")
second = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_347154_CII_A_Response_NSUB4.npz")

ks, Pks, db = second["k"], second["Pkmean"], second["deltab"]
ks = ks

# Set up colormap normalization
norm = colors.Normalize(vmin=np.min(db), vmax=np.max(db))
cmap = cm.cividis
sm = cm.ScalarMappable(norm=norm, cmap=cmap)  # for the colorbar

# Plot
fig, ax = plt.subplots()
for i in range(Pks.shape[0]):
    color = cmap(norm(db[i]))
    ax.loglog(ks, Pks[i, :], color=color)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")

plt.loglog(pkfile[:,0], pkfile[:, 1], "k--")


ax.set_title(r"[CII], $z$=1")
ax.set_xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
ax.set_ylabel(r"$P(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
fig.tight_layout()
plt.show()

In [ ]:
Pkmean = np.mean(Pks, axis=0)
dPk = Pks - Pkmean
print(dPk.shape)

In [ ]:
cov_w_SSC = 1/(len(Pkmean)-1) * np.sum(dPk[:, :, None] * dPk[:, None, :], axis=0)

In [ ]:
np.diag(cov_w_SSC)

In [ ]:
imin = np.argmin(np.abs(dTb))
plt.loglog(kc_BB, np.diag(cov_BB) * 4**3, label="rescaled Big Box")
plt.loglog(k, dCov[imin, :], label="minimum $\delta T_\mathrm{b}$ subbox")
plt.loglog(kSSC, np.diag(cov_w_SSC), label="cross-subbox variance")
plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}_{ii}\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")
plt.legend()

In [ ]:
nsub4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_346009_subdivided_JK.npz")
nsub3 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_348254_subdivided_JK.npz")
nsub2 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_348253_subdivided_JK.npz")

In [ ]:
kc_BB.shape
cov_BB.shape

In [ ]:
from scipy.interpolate import RectBivariateSpline

In [ ]:
k1 = 1

inter = RectBivariateSpline(kc_BB, kc_BB, cov_BB)
plt.loglog(kc_BB, inter(k1, kc_BB, grid=False), label="Big Box")

for i, f in zip([2,3,4], [nsub2, nsub3, nsub4]):
    kc = np.sqrt(f["kedges"][1:] * f["kedges"][:-1])

    imin = np.argmin(np.abs(f["deltab"]))
    cov_i = f["Cov"][0, :, :]

    covclean = cov_i[~np.isnan(cov_i)]
    covclean = covclean.reshape(int(np.sqrt(len(covclean))), int(np.sqrt(len(covclean))))
    kclean = kc[~np.isnan(cov_i[:, 1])]
    inter = RectBivariateSpline(kclean, kclean, covclean, s=0)
    plt.loglog(kclean, inter(k1, kclean, grid=False) / i**3, label=f"Npart={i}")

plt.xlabel(r"$k_2\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k_1, k_2)\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")
plt.legend()
plt.title(r"Rescaled jackknife, $k_1={{{}}}\,h\,\mathrm{{Mpc}}^{{-1}}$".format(k1))


In [ ]:
inter = RectBivariateSpline(kc_BB, kc_BB, cov_BB)
plt.loglog(kc_BB, inter(kc_BB, kc_BB, grid=False), label="Big Box")

for i, f in zip([2,3,4], [nsub2, nsub3, nsub4]):
    kc = np.sqrt(f["kedges"][1:] * f["kedges"][:-1])

    imin = np.argmin(np.abs(f["deltab"]))
    cov_i = f["Cov"][4, :, :]

    covclean = cov_i[~np.isnan(cov_i)]
    covclean = covclean.reshape(int(np.sqrt(len(covclean))), int(np.sqrt(len(covclean))))
    kclean = kc[~np.isnan(cov_i[:, 1])]
    plt.loglog(kclean, np.diag(covclean) / i**3, label=f"Npart={i}")

plt.xlabel(r"$k_1\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k_1, k_1)\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")
plt.legend()
plt.title(r"Rescaled jackknife diagonal")
plt.ylim(1e-4, 1e5)

In [ ]:
plt.scatter(f["deltab"], f["Cov"][:, 15, 12])

In [ ]:
plt.errorbar(ki, ri, yerr=si, ls="--")
plt.semilogx()

In [ ]:
imin = np.argmin(np.abs(dTb))
cov_min_background = Covs[imin, :, :]

In [ ]:
from scipy.interpolate import RectBivariateSpline
bb_interp = RectBivariateSpline(kc_BB, kc_BB, cov_BB, s=0)

In [ ]:
plt.loglog(kc_BB, bb_interp(kc_BB, k[10]) / 2 * 4**3,ls="--", label="(approximate) rescaled Big Box")
plt.loglog(k, cov_min_background[:, 10], label="minimum $\delta T_\mathrm{b}$ subbox")
plt.loglog(kSSC, cov_w_SSC[:, 10], label="cross-subbox variance")
plt.xlabel(r"$k_2\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k_1, k_2)\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")
plt.title(r"$k_1=0.31\,h\,\mathrm{Mpc}^{-1}$")
plt.legend()

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

In [ ]:
# Parameters for the Survey specifications
z = np.array([1])
def get_specs(nu):
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.05143

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 18.64 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

# [CII]

In [ ]:
astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
    "meanperserve_scatter": False,
}

nu = 1.897 * u.THz
surveyspecs_CII = get_specs(nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

In [ ]:
pobs_CII = spobs.PowerSpectra(myastro)
ssc_CII = scov.SuperSampleCovariance(pobs_CII)

In [ ]:
def sigma_survey(self):
    k = self.kgrid
    mu = self.mu
    z = np.atleast_1d(self.z)

    V = self.survey_specs.Vfield()
    W = self.survey_specs.Wsurvey(self.kgrid, self.mu)
    W = np.reshape(W, (*k.shape, *mu.shape, *z.shape))

    P = np.reshape(
        self.cosmology.matpow(k, z, nonlinear=False, tracer=self.halomodel.tracer),
        (*k.shape, *z.shape))

    sigma2_intgrnd = (
        2 * np.pi
        * (self.kgrid[:, None, None] / (2 * np.pi))**3
        * W**2
        * P[:, None, :]
    )
    sigma2_intgrnd = np.trapz(sigma2_intgrnd, x=mu, axis=1)
    plt.loglog(k, sigma2_intgrnd)
sigma_survey(ssc_CII)

In [ ]:
Cov_SSC_CII = ssc_CII.compute_SSC().to(myastro.Mpch**6 * u.uK**4).value[:, :, 0]
r_direct = ssc_CII.response(ki * myastro.Mpch**-1, 1).value / np.exp(mypk(np.log((ki * myastro.Mpch**-1).value)))

In [ ]:
plt.semilogx(ki, r_direct)
plt.semilogx(ki, ri)

In [ ]:
plt.loglog(ssc_CII.k, np.diag(Cov_SSC_CII))

In [ ]:
ssc_CII.sigma_survey() / ssc_CII.survey_specs.Vfield()

In [ ]:
def f(deltaF):
    nu = myastro.nu
    z = 1
    nuObs = nu / (1 + z)
    nuM = nuObs * (1 - deltaF / 2)
    nuP = nuObs * (1 + deltaF / 2)

    zmin = (nu / nuP).to(1).value - 1
    zmax = (nu / nuM).to(1).value - 1
    l = mycosmo.comoving([zmin, zmax])
    return np.diff(l)


In [ ]:
f(0.20544)

In [ ]:
f(0.05143)

In [ ]:
(((1024/4 * u.Mpc)**2 / mycosmo.comoving(1)**2) * u.rad**2).to(u.deg**2)

In [ ]:
pk_az = np.loadtxt("/home/sefa/Desktop/LIM-Code/clusterdata/pkfliles_from_source/snapshot_a0.500/ModelCII_A_N1024/ModelCII_A_pks_1d_0.5000.txt")
k_az, Pk_az = pk_az[:, 0], pk_az[:, 1]

k_az = (k_az * myastro.Mpch**-1).to(myastro.Mpch**-1)
Pk_az = (Pk_az * myastro.Mpch**3 * u.uK**2).to(myastro.Mpch**3 * u.uK**2)

I11 = myastro.Thalo(z, k_az, p=1, scale=(1,), beta=1)
I02 = myastro.Thalo(z, k_az, p=1, scale=(2,), beta=0)
Pk_Lim_Solo = (I11**2) * mycosmo.matpow(k_az, z) + I02

plt.loglog(k_az, Pk_az, c=Cp[0], label="Sims")
plt.loglog(k_az, Pk_Lim_Solo.to(myastro.Mpch**3 * u.uK**2), c=Cp[1], ls="--", label="SSLimPy")

plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
plt.title(r"[CII], $z=1$")
plt.legend()

In [ ]:
pk_az = np.loadtxt("/home/sefa/Desktop/LIM-Code/clusterdata/pkfliles_from_source/snapshot_a0.3333/ModelCII_A_N1024/ModelCII_A_pks_1d_0.3333.txt")
k_az, Pk_az = pk_az[:, 0], pk_az[:, 1]

k_az = (k_az * myastro.Mpch**-1).to(myastro.Mpch**-1)
Pk_az = (Pk_az * myastro.Mpch**3 * u.uK**2).to(myastro.Mpch**3 * u.uK**2)

I11 = myastro.Thalo(2, k_az, p=1, scale=(1,), beta=1)
I02 = myastro.Thalo(2, k_az, p=1, scale=(2,), beta=0)
Pk_Lim_Solo = (I11**2) * mycosmo.matpow(k_az, 2) + I02

plt.loglog(k_az, Pk_az, c=Cp[0], label="Pk files")
plt.loglog(k_az, Pk_Lim_Solo.to(myastro.Mpch**3 * u.uK**2), c=Cp[1], ls="--", label="SSLimPy")

plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
plt.title(r"[CII], $z=2$")
plt.legend()

In [ ]:
second = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_189352_CII_A_Response_NSUB8.npz")
ks, Pks, db = second["k"], second["Pkmean"], second["deltab"]
ks = (ks * myastro.Mpch**-1).to(myastro.Mpch**-1)

# Set up colormap normalization
norm = colors.Normalize(vmin=np.min(db), vmax=np.max(db))
cmap = cm.cividis
sm = cm.ScalarMappable(norm=norm, cmap=cmap)  # for the colorbar

# Plot
fig, ax = plt.subplots()
for i in range(Pks.shape[0]):
    color = cmap(norm(db[i]))
    ax.loglog(ks[i, :], Pks[i, :], color=color)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")

ax.set_title(r"[CII], $z$=1")
ax.set_xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
ax.set_ylabel(r"$P(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
fig.tight_layout()
plt.show()

In [ ]:
def linear(x, a, b):
    return a * x + b

pobs, pcov = [], []
for i in range(Pks.shape[1]):
    pi, ci = curve_fit(linear, db, Pks[:, i])
    pobs.append(pi)
    pcov.append(ci)
pobs = np.array(pobs)
pcov = np.array(pcov)

ai, bi = pobs[:, 0], pobs[:, 1]

ki = ks[0, :]
ri = myastro.Tavg(1).to(u.uK).value * ai / bi
si = ri * np.sqrt(pcov[:, 0, 0]/ ai**2 + pcov[:, 1, 1]/ bi**2 - 2 * bi / ai * pcov[:, 0, 1])

In [ ]:
kr = np.geomspace(7e-2, 3) * myastro.Mpch**-1

Delta = 4 * np.pi / (2 * np.pi)**3 * mycosmo.k**3 * mycosmo.matpow(mycosmo.k, 1, myastro.halomodel.tracer)
logD = np.log(Delta.to(1).value)

gamma = UnivariateSpline(np.log(mycosmo.k.value), logD, s=0).derivative(1)(np.log(kr.to(mycosmo.k.unit).value))
Pk = myastro.cosmology.matpow(kr, 1, myastro.halomodel.tracer)

In [ ]:
I11 = myastro.Thalo(1, kr, p=1, scale=(1,), beta=1)
I02 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=0)
I21 = (
    myastro.Thalo(1, kr, p=1, scale=(1,), beta="b2")
    + 4 / 3 * myastro.Thalo(1, kr, p=1, scale=(1,), beta="bG2")
)
I12 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=1)

Phalo = I11**2 * Pk + I02

BC = (68 / 21 * I11**2 + 2 * I11 * I21) * Pk
LD = - 1 / 3 * gamma * I11**2 * Pk
HSV = I12
response_ssl = (BC + LD + HSV) / Phalo

In [ ]:
plt.errorbar(ki.to(u.Mpc**-1), ri, si, ls="")
plt.semilogx(kr, (BC + LD + HSV) / Phalo, "k")
plt.xlabel("$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel("$\mathrm{dlog}P\,/\,\mathrm{d}\delta_\mathrm{b}$")
plt.title("[CII], $z$=1")

# CO2-1

In [ ]:
# Parameters for Astrophysics.Cov_SSC_CII.shape
astrodict_CO21={
    "model_type": "ML",
    "model_name": "TonyLi",
    "model_par": {
        "alpha": 1.11,
        "beta": 0.6,
        "dMF": 1 * u.Msun * u.yr**-1 * u.Lsun**-1,
        "sig_SFR":0,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
    "meanperserve_scatter": False,
} # CO21

nu = 2 * 115.27 * u.GHz # CO(2-1)
surveyspecs_CO21 = get_specs(nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CO21,
    obspars_dict=surveyspecs_CO21,
)

myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

In [ ]:
mithral_CII_Pk = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_182753_CO21_Powerspectrum.npz")
k, Pk = mithral_CII_Pk["k"], mithral_CII_Pk["Pk"]
k = (k * myastro.Mpch**-1).to(myastro.Mpch**-1)
Pk = (Pk * myastro.Mpch**3 * u.uK**2).to(myastro.Mpch**3 * u.uK**2)

I11 = myastro.Thalo(1, k, p=1, scale=(1,), beta=1)
I02 = myastro.Thalo(1, k, p=1, scale=(2,), beta=0)
Pk_Lim_Solo = (I11**2) * mycosmo.matpow(k, 1) * 0.8 + I02

fig, axs = plt.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [5, 2]})

axs[0].loglog(k, Pk, label="Pk files", c=Cp[0])
axs[0].loglog(k, Pk_Lim_Solo.to(myastro.Mpch**3 * u.uK**2), c=Cp[1],ls="--", label="SSLimPy")
axs[1].semilogx(k, ((Pk_Lim_Solo / Pk).to(1).value -1) * 100, c=Cp[1])
axs[0].legend()
axs[1].set_xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
axs[0].set_ylabel(r"$P(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
axs[1].set_ylabel(r"\% deviation")
axs[0].set_title("CO2-1, $z$=1")
axs[1].set_ylim(-10, 50)

plt.tight_layout()
fig.subplots_adjust(hspace=0.0)

In [ ]:
second = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_183794_CO21_Response_NSUB8.npz")
ks, Pks, db = second["k"], second["Pkmean"], second["deltab"]
ks = (ks * myastro.Mpch**-1).to(myastro.Mpch**-1)

In [ ]:
# Set up colormap normalization
norm = colors.Normalize(vmin=np.min(db), vmax=np.max(db))
cmap = cm.cividis
sm = cm.ScalarMappable(norm=norm, cmap=cmap)  # for the colorbar

# Plot
fig, ax = plt.subplots()
for i in range(Pks.shape[0]):
    color = cmap(norm(db[i]))
    ax.loglog(ks[i, :], Pks[i, :], color=color)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")

ax.set_title(r"CO(2-1), $z$=1")
ax.set_xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
ax.set_ylabel(r"$P(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
fig.tight_layout()
plt.show()

In [ ]:
def linear(x, a, b):
    return a * x + b

pobs, pcov = [], []
for i in range(Pks.shape[1]):
    pi, ci = curve_fit(linear, db, Pks[:, i])
    pobs.append(pi)
    pcov.append(ci)
pobs = np.array(pobs)
pcov = np.array(pcov)

ai, bi = pobs[:, 0], pobs[:, 1]

ki = ks[0, :]
ri = myastro.Tavg(1).to(u.uK).value * ai / bi
si = ri * np.sqrt(pcov[:, 0, 0]/ ai**2 + pcov[:, 1, 1]/ bi**2 - 2 * bi / ai * pcov[:, 0, 1])

In [ ]:
kr = np.geomspace(7e-2, 3) * myastro.Mpch**-1

Delta = 4 * np.pi / (2 * np.pi)**3 * mycosmo.k**3 * mycosmo.matpow(mycosmo.k, 1, myastro.halomodel.tracer)
logD = np.log(Delta.to(1).value)

gamma = UnivariateSpline(np.log(mycosmo.k.value), logD, s=0).derivative(1)(np.log(kr.to(mycosmo.k.unit).value))
Pk = myastro.cosmology.matpow(kr, 1, myastro.halomodel.tracer)

In [ ]:
I11 = myastro.Thalo(1, kr, p=1, scale=(1,), beta=1)
I02 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=0)
I21 = (
    myastro.Thalo(1, kr, p=1, scale=(1,), beta="b2")
    + 4 / 3 * myastro.Thalo(1, kr, p=1, scale=(1,), beta="bG2")
)
I12 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=1)

Phalo = I11**2 * Pk + I02

In [ ]:
BC = (68 / 21 * I11**2 + 2 * I11 * I21) * Pk
LD = - 1 / 3 * gamma * I11**2 * Pk
HSV = I12

In [ ]:
plt.semilogx(kr, (BC + LD + HSV) / Phalo, "k")
plt.errorbar(ki.to(u.Mpc**-1), ri, si, ls="")
plt.xlabel("$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel("$\mathrm{dlog}P\,/\,\mathrm{d}\delta_\mathrm{b}$")
plt.title("CO(2-1), $z$=1")